In [19]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, cross_validate

import statsmodels.api as sm

1. Przygotowanie

In [18]:
ames = pd.read_csv("AMES_housing_Price.csv", delimiter=",")
ames.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [20]:
subset = [ 'SalePrice','GrLivArea','LotArea','FullBath','BedroomAbvGr','OverallQual']

ames[subset].head()

,SalePrice,GrLivArea,LotArea,FullBath,BedroomAbvGr,OverallQual
0,208500,1710,8450,2,3,7
1,181500,1262,9600,2,3,6
2,223500,1786,11250,2,3,7
3,140000,1717,9550,1,3,7
4,250000,2198,14260,2,4,8


In [23]:
df = ames[subset].copy()

missing_values = ["?", "n/a", "N/A", "na", "NA", "brak", "Brak", "--", "-", "", "None", "null", "NULL", "很好"]

df = df.replace(missing_values, np.nan)
df = df.apply(pd.to_numeric, errors='coerce')

df.isna().sum()

,0
SalePrice,0
GrLivArea,0
LotArea,0
FullBath,0
BedroomAbvGr,0
OverallQual,0


In [24]:
df = df.fillna(df.median(numeric_only=True))
df.isna().sum()

,0
SalePrice,0
GrLivArea,0
LotArea,0
FullBath,0
BedroomAbvGr,0
OverallQual,0


In [25]:
predictors = ['GrLivArea', 'LotArea', 'FullBath', 'BedroomAbvGr', 'OverallQual']
outcome = 'SalePrice'

X = df[predictors]
y = df[outcome]

2. Model bazowy

In [28]:
ames_lm = LinearRegression()
ames_lm.fit(X, y)

print(f'Intercept: {ames_lm.intercept_:.3f}')
print('Coefficients:')
for name, coef in zip(predictors, ames_lm.coef_):
    print(f' {name}: {coef}')

fitted = ames_lm.predict(X)
RMSE_all = np.sqrt(mean_squared_error(y, fitted))
r2 = r2_score(y, fitted)
print(f'RMSE: {RMSE_all:.0f}')
print(f'r2: {r2:.4f}')

Intercept: -80928.194
Coefficients:
 GrLivArea: 60.69621535071177
 LotArea: 0.8859982301992968
 FullBath: 7126.642946299955
 BedroomAbvGr: -12260.164537087698
 OverallQual: 30255.51344054459
RMSE: 40725
r2: 0.7370


In [29]:
X_sm = sm.add_constant(X)
y_sm = y

model_sm = sm.OLS(y_sm, X_sm).fit()
model_sm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SalePrice   R-squared:                       0.737
Model:                            OLS   Adj. R-squared:                  0.736
Method:                 Least Squares   F-statistic:                     815.0
Date:                Wed, 26 Nov 2025   Prob (F-statistic):               0.00
Time:                        18:54:25   Log-Likelihood:                -17569.
No. Observations:                1460   AIC:                         3.515e+04
Df Residuals:                    1454   BIC:                         3.518e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const        -8.093e+04   6185.185    -13.084      0.000   -9.31e+04   -6.88e+04
GrLivArea       60.6962      3.340     18.173      0.000      54.145      67.248
LotArea          0.8860      0.111      7.958      0.000       0.668       1.104
FullBath      7126.6429   2635.094      2.705      0.007    1957.650    1.23e+04
BedroomAbvGr -1.226e+04   1629.587     -7.523      0.000   -1.55e+04   -9063.571
OverallQual   3.026e+04   1061.986     28.490      0.000    2.82e+04    3.23e+04
==============================================================================
Omnibus:                      359.868   Durbin-Watson:                   1.984
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            19846.239
Skew:                          -0.114   Prob(JB):                         0.00
Kurtosis:                      21.061   Cond. No.                     8.60e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 8.6e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_train = LinearRegression()
model_train.fit(X_train, y_train)

rmse_train = np.sqrt(mean_squared_error(y_train, model_train.predict(X_train)))
rmse_test = np.sqrt(mean_squared_error(y_test, model_train.predict(X_test)))

print(f'RMSE training: {rmse_train:.0f}')
print(f'RMSE test: {rmse_test:.0f}')

RMSE training: 40433
RMSE test: 42244


In [38]:
cv_results_baseline = cross_validate(LinearRegression(), X, y, cv=5, scoring='neg_mean_squared_error', return_train_score=True)

test_scores_baseline = np.sqrt(-cv_results_baseline['test_score'])
train_scores_baseline = np.sqrt(-cv_results_baseline['train_score'])

print("Baseline – RMSE train (foldy):", np.round(train_scores_baseline, 0))
print("Baseline – RMSE test  (foldy):", np.round(test_scores_baseline, 0))
print(f"\nBaseline – średni test RMSE: {test_scores_baseline.mean():.0f}")

Baseline – RMSE train (foldy): [42049. 40738. 40119. 41631. 38684.]
Baseline – RMSE test  (foldy): [34997. 41198. 43608. 36974. 48571.]

Baseline – średni test RMSE: 41070


3. Selekcja

In [43]:
predictors_1 = ['GrLivArea', 'OverallQual']
X1 = df[predictors_1]

predictors_2 = ['GrLivArea', 'LotArea', 'FullBath', 'BedroomAbvGr', 'OverallQual']
X2 = df[predictors_2]

predictors_3 = ['GrLivArea', 'LotArea', 'BedroomAbvGr', 'OverallQual']
X3 = df[predictors_3]

predictors_4 = ['GrLivArea', 'LotArea', 'OverallQual']
X4 = df[predictors_4]

In [45]:
cv_scores_1 = cross_val_score(
    LinearRegression(), X1, y,
    cv=5, scoring='neg_mean_squared_error'
)
rmse_1 = np.sqrt(-cv_scores_1.mean())

cv_scores_2 = cross_val_score(
    LinearRegression(), X2, y,
    cv=5, scoring='neg_mean_squared_error'
)
rmse_2 = np.sqrt(-cv_scores_2.mean())

cv_scores_3 = cross_val_score(
    LinearRegression(), X3, y,
    cv=5, scoring='neg_mean_squared_error'
)
rmse_3 = np.sqrt(-cv_scores_3.mean())

cv_scores_4 = cross_val_score(
    LinearRegression(), X4, y,
    cv=5, scoring='neg_mean_squared_error'
)
rmse_4 = np.sqrt(-cv_scores_4.mean())

print(f'Model 1 (mało cech)   - RMSE: {rmse_1:.0f}')
print(f'Model 2 (więcej cech) - RMSE: {rmse_2:.0f}')
print(f'Model 3 (bez FullBath) - RMSE: {rmse_3:.0f}')
print(f'Model 4 (dodatkowo bez BedroomAbvGr) - RMSE: {rmse_4:.0f}')

Model 1 (mało cech)   - RMSE: 42791
Model 2 (więcej cech) - RMSE: 41352
Model 3 (bez FullBath) - RMSE: 41411
Model 4 (dodatkowo bez BedroomAbvGr) - RMSE: 42015


4. Ocena i decyzja

In [46]:
best_predictors = predictors_2
X_best = df[best_predictors]

cv_results_best = cross_validate(
    LinearRegression(), X_best, y,
    cv=5,
    scoring='neg_mean_squared_error',
    return_train_score=True
)

test_scores = np.sqrt(-cv_results_best['test_score'])
train_scores = np.sqrt(-cv_results_best['train_score'])

print("RMSE dla każdego fold'u:")
for i, (train, test) in enumerate(zip(train_scores, test_scores)):
    print(f'Fold {i+1}: train={train:.0f}, test={test:.0f}')

print(f'\nŚrednia test RMSE: {test_scores.mean():.0f}')
print(f'Odchylenie std: {test_scores.std():.0f}')

RMSE dla każdego fold'u:
Fold 1: train=42049, test=34997
Fold 2: train=40738, test=41198
Fold 3: train=40119, test=43608
Fold 4: train=41631, test=36974
Fold 5: train=38684, test=48571

Średnia test RMSE: 41070
Odchylenie std: 4825


5. Weryfikacja

Wybrany model korzysta z predykatów:
* 'GrLivArea', czyli powierzchnia mieszkalna
* 'LotArea', czyli powierzchnia działki
* 'FullBath', czyli liczba łazienek
* 'OverallQual', czyli ogólna jakość domu
* 'BedroomAbvGr', czyli liczba sypialni

* Czy zmienne mają sens biznesowy?
  * Według mnie zmienne zdecyfowanie mają sens, szczególnie w przypadku 4 pierwszych z nich. Jasne jest, że większa powierzchnia mieszkania czy działkia wraz z większą ilością łazienek zwiększać będzie cene domu. Sprawa wygląda identycznie a nawet najbardziej jeżeli chodzi o jakoś domu. Jedyne wątpliwości może budzić ostatnia zmienna dotycząca ilości sypialni, której ujemny współczynnik może oznaczać że im więcej sypialni tym atrakcyjność budynku mniejsza.
* Czy model jest prosty i zrozumiały?
  * Tak, gdyż wykorzystuje jedynie 5 intuicyjnych cech. Z analizy train vs test RMSE można wywnioskowac, że są one dość zbliżone a różnice nie wskazują na przefitowanie. Model wydaje się uczyć stabilnie.
* Czy będzie działać na nowych danych?
  * Według mnie tak, gdyż opiera się on na typowych cech kształtująćych ceny nieruchomości i powinien on działać dość dobrze na nowych, niewidzianych jeszcze danych.



6. Finalizacja

In [35]:
X_final = df[best_predictors]
y_final = y

final_model = LinearRegression()
final_model.fit(X_final, y_final)

rmse_final_all = np.sqrt(mean_squared_error(y_final, final_model.predict(X_final)))
print(f'Finalny model – RMSE (all data): {rmse_final_all:.0f}')

Finalny model – RMSE (all data): 40725


In [36]:
print("Intercept:", final_model.intercept_)
print("Współczynniki:")
for name, coef in zip(best_predictors, final_model.coef_):
    print(f' {name}: {coef}')

Intercept: -80928.19367406695
Współczynniki:
 GrLivArea: 60.69621535071177
 LotArea: 0.8859982301992968
 FullBath: 7126.642946299955
 BedroomAbvGr: -12260.164537087698
 OverallQual: 30255.51344054459
